In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!

from pyspark.sql import functions as F

dim_date = spark.table("dbo.dim_date")
dim_store = spark.table("dbo.dim_store")
dim_product = spark.table("dbo.dim_product")
dim_customer = spark.table("dbo.dim_customer")
fact_sales = spark.table("dbo.fact_sales")
fact_online = spark.table("dbo.fact_online_orders")

print("Silver tables loaded.")



StatementMeta(, 952f24bd-f6e9-4c0d-8404-971c9eb7eb99, 3, Finished, Available, Finished, False)

Silver tables loaded.


In [2]:
instore = (
    fact_sales
    .join(dim_date, "date_key")
    .groupBy("full_date", "day_name", "month_name", "year_month", "is_weekend")
    .agg(
        F.countDistinct("transaction_id").alias("total_orders"),
        F.sum("quantity").alias("total_items"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_order_value")
    )
    .withColumn("channel", F.lit("In-Store"))
)

online = (
    fact_online
    .join(dim_date, "date_key")
    .groupBy("full_date", "day_name", "month_name", "year_month", "is_weekend")
    .agg(
        F.countDistinct("order_id").alias("total_orders"),
        F.sum("quantity").alias("total_items"),
        F.round(F.sum("line_total"), 2).alias("revenue"),
        F.round(F.avg("line_total"), 2).alias("avg_order_value")
    )
    .withColumn("channel", F.lit("Online"))
)

gold_daily_sales = instore.unionByName(online)

gold_daily_sales.write.mode("overwrite").saveAsTable("dbo.gold_daily_sales")

print("gold_daily_sales created.")


StatementMeta(, 952f24bd-f6e9-4c0d-8404-971c9eb7eb99, 4, Finished, Available, Finished, False)

gold_daily_sales created.


In [3]:
gold_store_performance = (
    fact_sales
    .join(dim_store, "store_key")
    .join(dim_date, "date_key")
    .groupBy(
        "store_id", "store_name", "city",
        "state", "store_type", "manager",
        "year_month"
    )
    .agg(
        F.countDistinct("transaction_id").alias("total_transactions"),
        F.sum("quantity").alias("items_sold"),
        F.round(F.sum("total_amount"), 2).alias("revenue"),
        F.round(F.avg("total_amount"), 2).alias("avg_basket_size")
    )
)

gold_store_performance.write.mode("overwrite").saveAsTable("dbo.gold_store_performance")

print("gold_store_performance created.")


StatementMeta(, 952f24bd-f6e9-4c0d-8404-971c9eb7eb99, 5, Finished, Available, Finished, False)

gold_store_performance created.


In [4]:
instore_totals = fact_sales.agg(
    F.round(F.sum("total_amount"),2).alias("instore_revenue"),
    F.countDistinct("transaction_id").alias("instore_orders")
)

online_totals = fact_online.agg(
    F.round(F.sum("line_total"),2).alias("online_revenue"),
    F.countDistinct("order_id").alias("online_orders")
)

gold_executive_summary = instore_totals.crossJoin(online_totals)

gold_executive_summary.write.mode("overwrite").saveAsTable("dbo.gold_executive_summary")

print("gold_executive_summary created.")


StatementMeta(, 952f24bd-f6e9-4c0d-8404-971c9eb7eb99, 6, Finished, Available, Finished, False)

gold_executive_summary created.
